In [3]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")


In [5]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from rich import print
import uuid

In [19]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from typing import Annotated
from operator import add

In [6]:
model = ChatOpenAI(
    model ='gpt-4.1-mini',
    api_key = api_key,
    temperature=0
)

In [20]:
class State(BaseModel):
    messages:Annotated[list[BaseMessage],add] = Field(..., description="stores all the conversation")
    original_problem_statement:str = Field(..., description="contains the original problem statement")
    result_in_transition:float = Field(..., description="contains the intermediatery result")
    final_result:float = Field(..., description="Contains the final output")

In [21]:
@tool
def add(*numbers):
    """THis tool sum up the value of all the arguments passed into it."""
    total = 0
    # Loop through the tuple of arguments
    for num in numbers:
        total += num
    return total

In [22]:
@tool
def substract(num1:float, num2:float)->float:
    """ This tool is used to find the difference between the two numbers"""
    return num1-num2

In [23]:
@tool
def multiply(*numbers)->float:
    """ This tool is used to multiply the values of all the arguments passed into it"""
    product=1
    # Loop through multiple arguments
    for num in numbers:
        product = product*num
    return product

In [24]:
@tool
def divide(num1:float, num2:float)->float:
    """THis tool is used to divide the argument passed into it."""
    try:
        if(num2!=0):
            return num1/num2
    except ZeroDivisionError:
        print("Can not divide by 0")

In [25]:
model_with_tools=model.bind_tools([add, substract, multiply, divide])